# HIDS Comprehensive Training
## Train on NSL-KDD + CICIDS2017 + CICIDS2018
### Fix 97% Anomaly Rate → 5-15%

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import os
import glob
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded")

✅ Libraries loaded


## 1. Load NSL-KDD (Best Normal Traffic)

In [2]:
def load_nslkdd():
    print("Loading NSL-KDD Dataset...")
    
    columns = ['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 
               'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 
               'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell',
               'su_attempted', 'num_root', 'num_file_creations', 'num_shells',
               'num_access_files', 'num_outbound_cmds', 'is_host_login',
               'is_guest_login', 'count', 'srv_count', 'serror_rate',
               'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate',
               'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count',
               'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate',
               'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate',
               'dst_host_serror_rate', 'dst_host_srv_serror_rate',
               'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty']
    
    train_df = pd.read_csv('/home/jovyan/datasets/KDDTrain+.txt', names=columns)
    test_df = pd.read_csv('/home/jovyan/datasets/KDDTest+.txt', names=columns)
    data = pd.concat([train_df, test_df], ignore_index=True)
    
    print(f"  Loaded: {len(data):,} samples")
    
    # Map to HIDS features
    X = pd.DataFrame({
        'src_port': 0,
        'dest_port': 0,
        'duration': data['duration'],
        'bytes_toserver': data['src_bytes'],
        'bytes_toclient': data['dst_bytes'],
        'pkts_toserver': data['count'],
        'proto': data['protocol_type'].map({'tcp': 1, 'udp': 2, 'icmp': 3}).fillna(1),
        'severity': 1
    })
    
    y = (data['label'] != 'normal').astype(int)
    
    print(f"  Normal: {(y==0).sum():,} ({(y==0).sum()/len(y)*100:.1f}%)")
    print(f"  Attack: {(y==1).sum():,} ({(y==1).sum()/len(y)*100:.1f}%)")
    
    return X, y

X_nslkdd, y_nslkdd = load_nslkdd()
print("\n✅ NSL-KDD loaded successfully")

Loading NSL-KDD Dataset...
  Loaded: 85,022 samples
  Normal: 42,902 (50.5%)
  Attack: 42,120 (49.5%)

✅ NSL-KDD loaded successfully


## 2. Load CICIDS2017

In [ ]:
def load_cicids2017():
    print("Loading CICIDS2017 Dataset...")
    
    # Use the cleaned CSV if available
    if os.path.exists('/home/jovyan/datasets/cicids2017_cleaned.csv'):
        print("  Using cleaned CICIDS2017 file")
        data = pd.read_csv('/home/jovyan/datasets/cicids2017_cleaned.csv', low_memory=False)
        data = data.sample(min(50000, len(data)), random_state=42)
    else:
        print("  ⚠️  CICIDS2017 ISCX files not found, skipping")
        return None, None
    
    data.columns = data.columns.str.strip()
    
    X = pd.DataFrame({
        'src_port': data.get('Source Port', 0),
        'dest_port': data.get('Destination Port', 0),
        'duration': data.get('Flow Duration', 0) / 1000000,
        'bytes_toserver': data.get('Total Length of Fwd Packets', 0),
        'bytes_toclient': data.get('Total Length of Bwd Packets', 0),
        'pkts_toserver': data.get('Total Fwd Packets', 1),
        'proto': 1,
        'severity': 1
    }).replace([np.inf, -np.inf], 0).fillna(0)
    
    label_col = [c for c in data.columns if 'label' in c.lower()][0]
    y = (data[label_col].str.strip().str.upper() != 'BENIGN').astype(int)
    
    print(f"  Loaded: {len(X):,} samples")
    print(f"  Normal: {(y==0).sum():,} ({(y==0).sum()/len(y)*100:.1f}%)")
    print(f"  Attack: {(y==1).sum():,} ({(y==1).sum()/len(y)*100:.1f}%)")
    
    return X, y

X_cicids17, y_cicids17 = load_cicids2017()
print("\n✅ CICIDS2017 loaded")

Loading CICIDS2017 Dataset...
  Using cleaned CICIDS2017 file


## 3. Load CICIDS2018

In [ ]:
import glob
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

def load_cicids2018():
    print("Loading CICIDS2018 Dataset...")
    
    csv_files = glob.glob('/home/jovyan/datasets/*2018*.csv')
    
    if not csv_files:
        print("  ⚠️  CICIDS2018 files not found, skipping")
        return None, None
    
    # Load first 2 files only (faster)
    dfs = []
    for file in tqdm(csv_files[:2], desc="Loading files"):
        try:
            df = pd.read_csv(file, low_memory=False)
            dfs.append(df)
        except:
            pass
    
    if not dfs:
        return None, None
    
    data = pd.concat(dfs, ignore_index=True).sample(min(30000, len(pd.concat(dfs))), random_state=42)
    
    X = pd.DataFrame({
        'src_port': data.get('Src Port', 0),
        'dest_port': data.get('Dst Port', 0),
        'duration': data.get('Flow Duration', 0) / 1000000,
        'bytes_toserver': data.get('Fwd Pkts Tot', 0) * 100,
        'bytes_toclient': data.get('Bwd Pkts Tot', 0) * 100,
        'pkts_toserver': data.get('Fwd Pkts Tot', 1),
        'proto': 1,
        'severity': 1
    }).replace([np.inf, -np.inf], 0).fillna(0)
    
    label_col = [c for c in data.columns if 'label' in c.lower()][0]
    y = (data[label_col].str.strip().str.upper() != 'BENIGN').astype(int)
    
    print(f"  Loaded: {len(X):,} samples")
    print(f"  Normal: {(y==0).sum():,} ({(y==0).sum()/len(y)*100:.1f}%)")
    print(f"  Attack: {(y==1).sum():,} ({(y==1).sum()/len(y)*100:.1f}%)")
    
    return X, y

X_cicids18, y_cicids18 = load_cicids2018()
print("\n✅ CICIDS2018 loaded")

## 4. Combine and Balance Datasets

In [ ]:
print("Combining datasets...")

X_all, y_all = [], []

for X, y in [(X_nslkdd, y_nslkdd), (X_cicids17, y_cicids17), (X_cicids18, y_cicids18)]:
    if X is not None:
        X_all.append(X)
        y_all.append(y)

X = pd.concat(X_all, ignore_index=True)
y = pd.concat(y_all, ignore_index=True)

print(f"\nCombined Total: {len(X):,} samples")
print(f"  Normal: {(y==0).sum():,} ({(y==0).sum()/len(y)*100:.1f}%)")
print(f"  Attack: {(y==1).sum():,} ({(y==1).sum()/len(y)*100:.1f}%)")

# Balance to 50/50
print("\nBalancing dataset to 50/50...")
normal = X[y == 0].sample(min(30000, (y==0).sum()), random_state=42)
attack = X[y == 1].sample(min(30000, (y==1).sum()), random_state=42)

X = pd.concat([normal, attack])
y = pd.concat([y[normal.index], y[attack.index]])

# Shuffle
idx = np.random.permutation(len(X))
X = X.iloc[idx].reset_index(drop=True)
y = y.iloc[idx].reset_index(drop=True)

print(f"\n✅ Final Balanced Dataset:")
print(f"  Normal: {(y==0).sum():,} ({(y==0).sum()/len(y)*100:.1f}%)")
print(f"  Attack: {(y==1).sum():,} ({(y==1).sum()/len(y)*100:.1f}%)")
print(f"  Total: {len(X):,} samples")

## 5. Train Random Forest Model

In [ ]:
print("Splitting train/test...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"  Training: {len(X_train):,} samples")
print(f"  Testing: {len(X_test):,} samples")

print("\nScaling features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n" + "="*70)
print("TRAINING RANDOM FOREST (This will take 2-5 minutes)")
print("="*70)

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_split=20,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=2  # Show progress
)

model.fit(X_train_scaled, y_train)

print("\n✅ Training complete!")

## 6. Evaluate Model

In [ ]:
print("Evaluating model...\n")

y_pred = model.predict(X_test_scaled)

print("="*70)
print("CLASSIFICATION REPORT")
print("="*70)
print(classification_report(y_test, y_pred, target_names=['Normal', 'Attack']))

cm = confusion_matrix(y_test, y_pred)
print("\n" + "="*70)
print("CONFUSION MATRIX")
print("="*70)
print(f"  True Negatives:  {cm[0,0]:,} (Correctly identified normal)")
print(f"  False Positives: {cm[0,1]:,} (Normal flagged as attack) ⚠️")
print(f"  False Negatives: {cm[1,0]:,} (Attack missed) ⚠️")
print(f"  True Positives:  {cm[1,1]:,} (Correctly identified attack)")

fpr = cm[0,1] / (cm[0,0] + cm[0,1]) * 100
print(f"\n📊 False Positive Rate: {fpr:.2f}%")
print(f"   Target: <5%")
print(f"   Status: {'✅ EXCELLENT' if fpr < 5 else '✅ GOOD' if fpr < 10 else '⚠️ NEEDS TUNING'}")

# Feature importance
print("\n" + "="*70)
print("FEATURE IMPORTANCE")
print("="*70)
features = ['src_port', 'dest_port', 'duration', 'bytes_toserver', 
            'bytes_toclient', 'pkts_toserver', 'proto', 'severity']
for feat, imp in sorted(zip(features, model.feature_importances_), 
                       key=lambda x: x[1], reverse=True):
    print(f"  {feat:20s}: {imp:.4f} {'█' * int(imp * 50)}")

## 7. Save Model

In [ ]:
print("Saving model...\n")

os.makedirs('/home/jovyan/work/models', exist_ok=True)

joblib.dump(model, '/home/jovyan/work/models/anomaly_detector.pkl')
joblib.dump(scaler, '/home/jovyan/work/models/scaler.pkl')

with open('/home/jovyan/work/models/model_type.txt', 'w') as f:
    f.write('random_forest_comprehensive')

print("✅ Model saved to /home/jovyan/work/models/")
print("   - anomaly_detector.pkl")
print("   - scaler.pkl")
print("   - model_type.txt")

## 8. Deploy to HIDS

In [ ]:
print("\n" + "="*70)
print("DEPLOYMENT COMMANDS")
print("="*70)
print("\nRun these commands in terminal:\n")
print("docker cp models/anomaly_detector.pkl hids_ml_service:/ap/home/jovyan/work/models/")
print("docker cp models/scaler.pkl hids_ml_service:/ap/home/jovyan/work/models/")
print("docker cp models/model_type.txt hids_ml_service:/ap/home/jovyan/work/models/")
print("docker compose restart ml_service hids_engine")
print("\n" + "="*70)
print("✅ TRAINING COMPLETE!")
print("="*70)
print("\nExpected Results:")
print("  • Anomaly rate: 5-15% (down from 97%)")
print("  • False positives: <5%")
print("  • Production ready!")